CNN-LSTM 混合网络
编码器-解码器 Seq2Seq（无注意力）

In [8]:
import torch
import torch.nn as nn 
class CNN_LSTM(nn.Module):
    def __init__(self):
      super().__init__()
      self.conv = nn.Sequential(
          nn.Conv1d(8,32,kernel_size=3,padding=1),nn.ReLU(),
          nn.Conv1d(32,32,kernel_size=3,padding=1),nn.ReLU(),
        )
      self.lstm = nn.LSTM(input_size=32,hidden_size=48,num_layers=1,batch_first=True)
      self.fc = nn.Linear(48,4)

    def forward(self,x):
        x = x.transpose(1,2)
        x = self.conv(x)
        x = x.transpose(1,2)
        out,(h,c) = self.lstm(x)
        return self.fc(h[-1])
model = CNN_LSTM()
model.eval()
with torch.no_grad():
    out = model(torch.randn(4,20,8))
    print(out.shape if isinstance(out,torch.Tensor) else type(out))

torch.Size([4, 4])


In [24]:
import torch
import torch.nn as nn
class EncoderDecoder(nn.Module):
    def __init__(self,enc_in=10,dec_in=8,hidden=32,out_steps=5,out_dim=3):
        super().__init__()
        self.out_steps = out_steps
        self.out_dim = out_dim
        self.enc = nn.LSTM(enc_in,hidden,batch_first=True)
        self.dec = nn.LSTM(dec_in,hidden,batch_first=True)
        self.head = nn.Linear(hidden,out_dim)
    def forward(self,src,tgt):
        _,(h,c) = self.enc(src)
        out,_ = self.dec(tgt,(h,c))
        return self.head(out)
model = EncoderDecoder()
model.eval()
with torch.no_grad():
    out = model(torch.randn(3,12,10),torch.randn(3,5,8))
    print(out)



tensor([[[ 0.1148,  0.1999, -0.0305],
         [ 0.1173,  0.1846,  0.0120],
         [ 0.0930,  0.1975, -0.0251],
         [ 0.1260,  0.1535,  0.0460],
         [ 0.0783,  0.1667,  0.0309]],

        [[ 0.1871,  0.1600, -0.0575],
         [ 0.1576,  0.1814, -0.0581],
         [ 0.0887,  0.1227, -0.0588],
         [ 0.0758,  0.1209,  0.0058],
         [ 0.0811,  0.1374, -0.0393]],

        [[ 0.1523,  0.1918, -0.0789],
         [ 0.1858,  0.1407, -0.0541],
         [ 0.1783,  0.1762, -0.0298],
         [ 0.1131,  0.1616, -0.0551],
         [ 0.0424,  0.1510, -0.0615]]])


In [25]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
class BahaAtt(nn.Module):
    def __init__(self,enc_in=10,dec_in=8,hidden=32,out_steps=4,out_dim=3):
        super().__init__()
        self.out_steps = out_steps
        self.enc = nn.LSTM(enc_in,hidden,batch_first=True,bidirectional=True)
        self.dec = nn.LSTM(dec_in,hidden,batch_first=True)
        self.w1 = nn.Linear(hidden * 2, hidden,bias=False)
        self.w2 = nn.Linear(hidden,hidden,bias=False)
        self.v = nn.Linear(hidden,1,bias=False)
        self.bridge_h = nn.Linear(hidden * 2,hidden)
        self.bridge_c = nn.Linear(hidden * 2,hidden)
        self.head = nn.Linear(hidden * 2 + hidden,out_dim)
    def forward(self,src,tgt):
        enc_out,(h,c) = self.enc(src)
        h_dec = torch.cat((h[-2],h[-1]),dim=1)
        c_dec = torch.cat((c[-2],c[-1]),dim=1)
        h0 = self.bridge_h(h_dec).unsqueeze(0)
        c0 = self.bridge_c(c_dec).unsqueeze(0)
        dec_out,_ = self.dec(tgt,(h0,c0))
        scores = self.v(torch.tanh(
            self.w1(enc_out).unsqueeze(1) + self.w2(dec_out).unsqueeze(2)
        )).squeeze(-1)
        attn = F.softmax(scores,dim=-1)
        ctx = torch.bmm(attn,enc_out)
        out = self.head(torch.cat([dec_out,ctx],dim=-1))
        return out
model = BahaAtt()
model.eval()
with torch.no_grad():
    out = model(torch.randn(3,12,10),torch.randn(3,4,8))
    print(out)

tensor([[[-0.0280,  0.0494,  0.0338],
         [-0.0106,  0.0616,  0.0831],
         [ 0.0226,  0.0336,  0.0649],
         [-0.0214,  0.0732,  0.1066]],

        [[ 0.0167,  0.1018,  0.0783],
         [-0.0064,  0.0976,  0.1013],
         [-0.0305,  0.1044,  0.0655],
         [-0.0223,  0.1057,  0.0608]],

        [[-0.0063,  0.0897,  0.0709],
         [-0.0241,  0.0616,  0.0225],
         [-0.0386,  0.0786,  0.0420],
         [-0.0255,  0.0547,  0.0403]]])
